# DistilBERT Fine-Tuning for Sentiment Classification
**Dataset:** IMDB 50K Movie Reviews (HuggingFace)  
**Model:** `distilbert-base-uncased` (HuggingFace Transformers + PyTorch)  
**Goal:** Fine-tune a pre-trained transformer and benchmark against classical and RNN baselines.

## 1. Imports & Configuration

In [1]:
import os
import json
import time
import numpy as np
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for CI
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from datasets import load_dataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
import evaluate
from sklearn.metrics import classification_report, confusion_matrix

import warnings
warnings.filterwarnings('ignore')

CI_MODE = os.environ.get('CI_MODE', 'false').lower() == 'true'

MODEL_CHECKPOINT = 'distilbert-base-uncased'
MAX_LEN = 128 if CI_MODE else 256
BATCH_SIZE = 16
EPOCHS = 1  if CI_MODE else 3
LEARNING_RATE = 2e-5
RANDOM_SEED = 42
TRAIN_SIZE = 3000 if CI_MODE else None   # None = full 25k
TEST_SIZE = 500  if CI_MODE else None

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'CI_MODE         : {CI_MODE}')
print(f'Device          : {device}')
print(f'PyTorch version : {torch.__version__}')
if device == 'cuda':
    print(f'GPU             : {torch.cuda.get_device_name(0)}')

CI_MODE         : True
Device          : cpu
PyTorch version : 2.11.0+cu130


## 2. Load Dataset

In [2]:
raw_dataset = load_dataset('imdb')

train_dataset = raw_dataset['train'].shuffle(seed=RANDOM_SEED)
test_dataset = raw_dataset['test'].shuffle(seed=RANDOM_SEED)

if TRAIN_SIZE:
    train_dataset = train_dataset.select(range(TRAIN_SIZE))
if TEST_SIZE:
    test_dataset = test_dataset.select(range(TEST_SIZE))

print(f'Train: {len(train_dataset):,} | Test: {len(test_dataset):,}')
print(f'\nSample:\n  text  : {train_dataset[0]["text"][:120]}...')
print(f'  label : {train_dataset[0]["label"]} ({"Positive" if train_dataset[0]["label"] else "Negative"})')

Train: 3,000 | Test: 500

Sample:
  text  : There is no relation at all between Fortier and Profiler but the fact that both are police series about violent crimes. ...
  label : 1 (Positive)


## 3. Tokenization

In [3]:
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_CHECKPOINT)

def tokenize_fn(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        padding='max_length',
        max_length=MAX_LEN
    )

tokenized_train = train_dataset.map(tokenize_fn, batched=True, batch_size=512)
tokenized_test = test_dataset.map(tokenize_fn,  batched=True, batch_size=512)

tokenized_train = tokenized_train.rename_column('label', 'labels')
tokenized_test = tokenized_test.rename_column('label', 'labels')

tokenized_train.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
tokenized_test.set_format('torch',  columns=['input_ids', 'attention_mask', 'labels'])

print('Tokenization complete.')
print(f'Sample token ids shape: {tokenized_train[0]["input_ids"].shape}')

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenization complete.
Sample token ids shape: torch.Size([128])


## 4. Load Pre-trained DistilBERT

In [4]:
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=2
)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Total parameters    : {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total parameters    : 66,955,010
Trainable parameters: 66,955,010


## 5. Define Metrics

In [5]:
accuracy_metric = evaluate.load('accuracy')
f1_metric = evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_metric.compute(predictions=preds, references=labels)['accuracy'],
        'f1'      : f1_metric.compute(predictions=preds, references=labels, average='binary')['f1']
    }

## 6. Training Arguments & Trainer

In [6]:
os.makedirs('results', exist_ok=True)

training_args = TrainingArguments(
    output_dir = './results/distilbert_checkpoints',
    num_train_epochs = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size = 32,
    learning_rate = LEARNING_RATE,
    warmup_ratio = 0.1,
    weight_decay = 0.01,
    eval_strategy = 'epoch',
    save_strategy = 'epoch',
    load_best_model_at_end = True,
    metric_for_best_model = 'f1',
    logging_steps = 50,
    fp16 = torch.cuda.is_available(),
    seed = RANDOM_SEED,
    report_to = 'none'
)

trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = tokenized_train,
    eval_dataset = tokenized_test,
    compute_metrics = compute_metrics,
    callbacks = [EarlyStoppingCallback(early_stopping_patience=2)]
)

print('Trainer ready.')

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Trainer ready.


## 7. Fine-tune

In [7]:
t0 = time.time()
train_result = trainer.train()
train_time = time.time() - t0

print(f'\nTraining time: {train_time/60:.1f} min')
print(f'Train loss   : {train_result.training_loss:.4f}')

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.387915,0.393194,0.812000,0.826568


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training time: 11.4 min
Train loss   : 0.4722


## 8. Training Curves

In [8]:
log_history = trainer.state.log_history
train_logs = [l for l in log_history if 'loss' in l and 'eval_loss' not in l]
eval_logs = [l for l in log_history if 'eval_loss' in l]

if train_logs and eval_logs:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot([l['step'] for l in train_logs], [l['loss'] for l in train_logs], label='Train Loss')
    axes[0].plot([l['step'] for l in eval_logs],  [l['eval_loss'] for l in eval_logs], 'o-', label='Val Loss')
    axes[0].set_title('DistilBERT, Loss')
    axes[0].set_xlabel('Step')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot([l['step'] for l in eval_logs], [l['eval_f1'] for l in eval_logs], 'o-', color='green', label='Val F1')
    axes[1].set_title('DistilBERT, Validation F1')
    axes[1].set_xlabel('Step')
    axes[1].set_ylabel('F1 Score')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.suptitle('DistilBERT Fine-tuning Curves', fontsize=14)
    plt.tight_layout()
    plt.savefig('results/bert_training_curves.png', dpi=150)
    plt.show()
else:
    print('Not enough log data to plot curves.')

## 9. Evaluation

In [9]:
eval_results = trainer.evaluate()
acc = eval_results['eval_accuracy']
f1 = eval_results['eval_f1']
print(f'Test Accuracy : {acc:.4f}')
print(f'Test F1       : {f1:.4f}')

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.387915,0.393194,1,0.812000,0.826568


Test Accuracy : 0.8120
Test F1       : 0.8266


In [10]:
single_text = tokenizer(
    "This movie was an absolute masterpiece.",
    return_tensors='pt', truncation=True, max_length=MAX_LEN
).to(device)

model.eval()
model.to(device)

with torch.no_grad():
    _ = model(**single_text)

times = []
with torch.no_grad():
    for _ in range(100):
        t0 = time.perf_counter()
        _ = model(**single_text)
        times.append((time.perf_counter() - t0) * 1000)

latency_ms = np.mean(times)
print(f'Inference latency (single sample): {latency_ms:.2f} ms (avg over 100 runs)')

Inference latency (single sample): 25.21 ms (avg over 100 runs)


In [11]:
pred_output = trainer.predict(tokenized_test)
preds = np.argmax(pred_output.predictions, axis=-1)
true_labels = pred_output.label_ids

print(classification_report(true_labels, preds, target_names=['Negative', 'Positive']))

cm = confusion_matrix(true_labels, preds)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'], ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('DistilBERT, Confusion Matrix')
plt.tight_layout()
plt.savefig('results/bert_confusion_matrix.png', dpi=150)
plt.show()

              precision    recall  f1-score   support

    Negative       0.89      0.72      0.79       254
    Positive       0.76      0.91      0.83       246

    accuracy                           0.81       500
   macro avg       0.82      0.81      0.81       500
weighted avg       0.83      0.81      0.81       500



## 10. Inference Demo

In [12]:
def predict_sentiment(text: str) -> dict:
    """Run inference on a raw review string."""
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=MAX_LEN).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]
    label = 'Positive' if probs[1] > 0.5 else 'Negative'
    return {'label': label, 'confidence': float(max(probs))}

samples = [
    "An absolute masterpiece. The performances were breathtaking and the story deeply moving.",
    "Terrible film. Boring plot, wooden acting, and a complete waste of two hours.",
    "It was okay. Some good moments but nothing particularly memorable."
]

for review in samples:
    result = predict_sentiment(review)
    print(f'Review    : {review[:60]}...')
    print(f'Prediction: {result["label"]} ({result["confidence"]:.2%} confidence)\n')

Review    : An absolute masterpiece. The performances were breathtaking ...
Prediction: Positive (90.08% confidence)



Review    : Terrible film. Boring plot, wooden acting, and a complete wa...
Prediction: Negative (92.43% confidence)

Review    : It was okay. Some good moments but nothing particularly memo...
Prediction: Positive (59.09% confidence)



## 11. Save Metrics

In [13]:
metrics = {
    'model'       : 'DistilBERT (distilbert-base-uncased)',
    'accuracy'    : round(acc, 4),
    'f1_score'    : round(f1, 4),
    'latency_ms'  : round(latency_ms, 2),
    'train_time_s': round(train_time, 2),
    'epochs'      : EPOCHS,
    'max_len'     : MAX_LEN,
    'device'      : device,
    'ci_mode'     : CI_MODE
}

with open('results/bert_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print('Metrics saved to results/bert_metrics.json')
print(json.dumps(metrics, indent=2))

Metrics saved to results/bert_metrics.json
{
  "model": "DistilBERT (distilbert-base-uncased)",
  "accuracy": 0.812,
  "f1_score": 0.8266,
  "latency_ms": 25.21,
  "train_time_s": 684.32,
  "epochs": 1,
  "max_len": 128,
  "device": "cpu",
  "ci_mode": true
}
